# Hubble-8B unlearning — GRADDIFF

Trains **GRADDIFF** from the Perturbed 8B checkpoint, pushes the model to HF, evaluates infill accuracy, and uploads results. Run the cells top to bottom on an **A100 (80GB)**.

> All four method notebooks are independent; the **master notebook** aggregates their HF results.

In [ ]:
# --- Setup: install deps, put the hubble8b package on the path, set HF token ---
import sys, os, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.53", "datasets", "accelerate", "bitsandbytes", "scipy"], check=False)

# Make `import hubble8b` work. Adjust HUB_DIR to wherever this repo lives on the box.
HUB_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
for cand in [HUB_DIR, os.path.join(HUB_DIR, "gpu_8b_run"), "/kaggle/input"]:
    if os.path.isdir(os.path.join(cand, "hubble8b")): sys.path.insert(0, cand); break

# HF token: NEVER hard-code. Prefer env; else Colab userdata; else prompt.
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        from getpass import getpass; os.environ["HF_TOKEN"] = getpass("HF token: ")

import hubble8b as h8
h8.hf_login()
print("device:", h8.DEVICE, "| target slot:", h8.TARGET_SLOT, "| LSR slot:", h8.LSR_SLOT)


In [ ]:
# --- Data: splits + retain + distractor pool (deterministic, seed=42) ---
forget_set, keep_set, unseen_set = h8.load_yago_splits()
retain = h8.load_retain_texts()
retain_texts = retain.select(range(min(len(retain), len(forget_set) * 40)))["text"]
splits = {"Unseen": unseen_set, "Unlearn": forget_set, "Keep": keep_set}
import json
pool_birth = h8.build_distractor_pool(forget_set, keep_set, unseen_set, h8.TARGET_SLOT)
print("birthdate distractors:", len(pool_birth))


In [ ]:
MODEL_NAME = "GRADDIFF"

In [ ]:
# --- Train GRADDIFF (inits from Perturbed; 8B needs A100 80GB) ---
import pandas as pd
model, hist = h8.TRAINERS["graddiff"](forget_set["text"], retain_texts)
h8.push_model(model, h8.get_tokenizer(), "graddiff")                 # -> HF model repo
h8.upload_artifact(  # training curve
    (lambda p: (pd.DataFrame(hist).to_csv(p, index=False) or p))(
        f"{h8.workdir()}/graddiff_train_history.csv"))


In [ ]:
# --- Evaluate this model: infill accuracy + push CSV to HF ---
tok = h8.get_tokenizer()
name = MODEL_NAME
df = h8.run_infill(model, name, splits, pool_birth, target_slot=h8.TARGET_SLOT)
import pandas as pd
p = f"{h8.workdir()}/infill_{name}.csv"; df.to_csv(p, index=False); h8.upload_artifact(p)
h8.to_cpu(model); del model; h8.free_cuda()
print(name, "done — infill CSV uploaded to HF.")
